In [10]:
import cv2
import time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import files
import imagezmq

In [3]:
!pip -q install imagezmq

In [4]:
stream_url = 'rtsp://wowzaec2demo.streamlock.net/vod/mp4:BigBuckBunny_115k.mp4'

print('Stream URL:', stream_url)

Stream URL: rtsp://wowzaec2demo.streamlock.net/vod/mp4:BigBuckBunny_115k.mp4


In [5]:
cap = cv2.VideoCapture(stream_url, cv2.CAP_FFMPEG)

if not cap.isOpened():
  print('RTSP Stream Could Not Be Opened.')
  print('If The Stream Is Blocked Or Unsupported, Use The Upload Fallback In The Next Section.')

else:
  frames = []
  frame_indices = []
  max_frames = 6
  frame_number = 0

  while len(frames) < max_frames:
    ret, frame = cap.read()

    if not ret:
      print('Stream Ended Or Frame Could Not Be Read.')
      break

    if frame_number % 10 == 0:
      frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
      frame_indices.append(frame_number)
    frame_number += 1

  cap.release()

  print('Frames captured:', len(frames))

RTSP Stream Could Not Be Opened.
If The Stream Is Blocked Or Unsupported, Use The Upload Fallback In The Next Section.


In [8]:
if 'frames' in globals() and len(frames) > 0:
  plt.figure(figsize=(18, 9))
  for i, (frame_rgb, index) in enumerate(zip(frames, frame_indices), start=1):
    plt.subplot(2, 3, i)
    plt.imshow(frame_rgb)
    plt.title(f'RTSP Frame {index}')
    plt.axis('off')

  plt.tight_layout()
  plt.show()

else:
  print('No RTSP Frames Are Available To Display.')

No RTSP Frames Are Available To Display.


In [ ]:
uploaded = files.upload()
video_name = next(iter(uploaded))
local_video_path = '/content/' + video_name

cap.cv2.VideoCapture(local_video_path)

if not cap.isOpened():
  raise ValueError('Local Video Could Not Be Opened.')

uploaded_frame = []
uploaded_frame_indices = []
frame_number = 0

while len(uploaded_frame) < 6:
  ret, frame = cap.read()

  if not ret:
    break

  if frame_number % 10 == 0:
    uploaded_frame.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    uploaded_frame_indices.append(frame_number)
  frame_number += 1

cap.release()

print('Uploaded video:', local_video_path)
print('Frames captured:', len(uploaded_frame_indices))

In [ ]:
plt.figure(figsize=(18, 9))
for i, (frame_rgb, index) in enumerate(zip(uploaded_frame, uploaded_frame_indices), start=1):
  plt.subplot(2, 3, i)
  plt.imshow(frame_rgb)
  plt.title(f'Uploaded Video Frame {index}')
  plt.axis('off')

plt.tight_layout()
plt.show()

In [11]:
if 'frames' in globals() and len(frames) > 0:
  snapshot_rgb = frames[-1]
  snapshot_bgr = cv2.cvtColor(snapshot_rgb, cv2.COLOR_RGB2BGR)
  snapshot_path = '/content/cctv_snapshot.jpg'

  cv2.imwrite(snapshot_path, snapshot_bgr)
  print('Saved Snapshot:', snapshot_path)

  plt.figure(figsize=(8, 6))
  plt.imshow(snapshot_rgb)
  plt.title('CCTV Snapshot')
  plt.axis('off')

  plt.show()

else:
  print('No Frames Available To Take A Snapshot.')

No Frames Available To Take A Snapshot.


In [ ]:
def receive_imagezmq_frames(max_frames=5):
  image_hub = imagezmq.ImageHub()
  received = []

  for _ in range(max_frames):
    rpi_name, image = image_hub.recv_image()
    received.append((rpi_name, cv2.cvtColor(image, cv2.COLOR_BGR2RGB)))

  return received